<a href="https://colab.research.google.com/github/harry-8818/-From-Neural-Network-Foundations-to-Multi-Object-Tracking/blob/main/FeatureExt_and_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import files
print("Please upload train.zip and test_set.zip...")
files.upload()
print("Extracting files...")
!unzip -q -o "train.zip" -d full_dataset/
!unzip -q -o "test_set.zip" -d full_dataset/
!find full_dataset/ -name ".DS_Store" -type f -delete
!rm -rf full_dataset/__MACOSX
!find full_dataset/ -name "__MACOSX" -type d -exec rm -rf {} +
print("Data extracted successfully!")

import copy
import csv
import torch
from torch import nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from PIL import Image

# Checking if the accelerator is available otherwise we will use CPU
device = ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using the device : {device}")

# Defining the transformation to be applied on the raw image to provide it to the Network
# Training Pipeline
# We can flip the images here because as we are using satellite images it doesn't matter anymore in which orientation te image was taken.
training_pipeline = v2.Compose([v2.RandomHorizontalFlip(p=0.5),v2.RandomVerticalFlip(p=0.5),v2.RandomRotation(degrees=30), # Spatial Changes
                                v2.ColorJitter(brightness=0.25,contrast=0.25,saturation=0.25), # Colour Changes
                                v2.ToImage(),v2.ToDtype(torch.float32, scale=True),v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])# Coversion of data using Normalisation

# Testing Pipeline
# No changes in data only conversion of data using Normalisation
testing_pipeline = v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True),v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

# To augment the training data and coverting the data using normalisation for the whole dataset
class data_changer(Dataset):
    def __init__(self,data,transform=None):
        self.data = data
        self.transform = transform
    def __getitem__(self,index):
        x,y = self.data[index]
        if self.transform:
            x = self.transform(x)
        return x,y
    def __len__(self):
        return len(self.data)

class testing_dataset(Dataset):
    def __init__(self,root_dir,transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir,img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image,img_name

# Initialize the model and ship to GPU
pre_trained_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for i in pre_trained_model.parameters():
    i.requires_grad = False

len_last_layer = pre_trained_model.fc.in_features
pre_trained_model.fc = nn.Linear(len_last_layer,10)
model = pre_trained_model.to(device)

full_training_data = ImageFolder(root="full_dataset/train")
class_names = full_training_data.classes
training_data_size = int(0.8 * len(full_training_data))
evaluation_data_size = len(full_training_data) - training_data_size
rs_training_data,rs_evaluation_data = random_split(full_training_data,[training_data_size,evaluation_data_size])

training_data = data_changer(rs_training_data,transform=training_pipeline)
evaluation_data = data_changer(rs_evaluation_data,transform=testing_pipeline)
test_data = testing_dataset(root_dir="full_dataset/test_set",transform=testing_pipeline)
train_dataloader = DataLoader(training_data,batch_size=64,shuffle=True)
evaluation_dataloader = DataLoader(evaluation_data,batch_size=64,shuffle=False)
test_dataloader = DataLoader(test_data,batch_size=64,shuffle=False)

epochs = 40
best_accuracy = 0.0
best_weights = None
optimizer = optim.Adam(pre_trained_model.fc.parameters(),lr=1e-3)
cost_fn = nn.CrossEntropyLoss()
learning_rate_changer = optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=2)


def train(dataloader,model,cost_fn,optimizer):
    model.train() # Turning the training specific operations on
    for batch,(images,labels) in enumerate(dataloader):
        images,labels = images.to(device),labels.to(device)
        predicted = model(images)
        cost = cost_fn(predicted,labels)
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

def evaluate(dataloader,model,cost_fn):
    model.eval() # Telling the model that the data is testing data so turn off training specific operations
    test_cost,correct = 0,0
    with torch.no_grad():
        for images,labels in dataloader:
            images,labels = images.to(device),labels.to(device)
            predicted = model(images)
            test_cost += cost_fn(predicted,labels).item()
            correct += (predicted.argmax(1) == labels).type(torch.float).sum().item()

    test_cost /= len(dataloader)
    accuracy = correct / len(dataloader.dataset)
    print(f"Evaluation Accuracy: {(100*accuracy):>0.1f}%, Average Cost: {test_cost:>8f}")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}")
    return accuracy

for i in range(epochs):
    print(f"\nExecuting Epoch Number : {i+1}")
    train(train_dataloader,model,cost_fn,optimizer)
    current_accuracy = evaluate(evaluation_dataloader,model,cost_fn)
    learning_rate_changer.step(current_accuracy)
    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        best_weights = copy.deepcopy(model.state_dict())

# Highest accuracy achieved and loading that weights of the model by saving them
print(f"\nHighest Recorded Validation Accuracy: {(100*best_accuracy):>0.1f}%")
if best_weights is not None:
    model.load_state_dict(best_weights)

print("\nGenerating final predictions for the test set")
model.eval()
results = []
class_mapping = {'AnnualCrop': 0,'Forest': 1,'HerbaceousVegetation': 2,'Highway': 3,'Industrial': 4,'Pasture': 5,'PermanentCrop': 6,'Residential': 7,'River': 8,'SeaLake': 9}
with torch.no_grad():
    for images,filenames in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(1).cpu().numpy()
        for i in range(len(filenames)):
            prediction = class_names[predictions[i]]
            label = class_mapping[prediction]
            results.append([filenames[i],label])
csv_file = "2025MT61788_FB.csv"
with open(csv_file,mode='w',newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['img_id','label'])
    writer.writerows(results)
files.download(csv_file)
print("Done")